# Gemini embeddings for utterances (side branch)

Parallel to `03_talk_generate_clip_embedding`, using `gemini-embedding-2` over the API.

**The chunking is gone.** The CLIP branch had to split transcripts at 77 tokens and
average the chunk vectors (`chunk_evenly_balanced`). `gemini-embedding-2` takes 8,192
tokens, so every transcript is embedded whole. 

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, ".")
import gemini_embed as ge

SEA_ANIMALS = ["crab", "octopus", "seahorse", "shark", "turtle", "whale"]
DIM = 768

TRANSCRIPTS_CSV = "../data/clean/transcripts_raw_clean.csv"
AGES_CSV = "../data/clean/participant_age_table.csv"

OUT_DIR = Path("../data/embeddings/gemini")
OUT_CSV = OUT_DIR / "utterance_gemini_embeddings.csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ge.CACHE_DIR = OUT_DIR / "_cache"
print("cache:", ge.cache_stats())

cache: {'vectors': 2271, 'json': 989, 'mb': 7.5}


In [2]:
utt = pd.read_csv(TRANSCRIPTS_CSV)
utt["participant_id"] = utt["participant_id"].astype(str).str.strip()
utt["target_category"] = utt["target_category"].astype(str).str.strip().str.lower()
utt = utt[utt["target_category"].isin(SEA_ANIMALS)].copy()

FULL_COL = "utterance_masked"
HALF_COL = "utterance_half_masked_filtered"

print(f"{len(utt)} rows")
print(f"  full non-empty: {utt[FULL_COL].notna().sum()}")
print(f"  half non-empty: {utt[HALF_COL].notna().sum()}")
utt[["participant_id", "target_category", FULL_COL]].head(3)

647 rows
  full non-empty: 547
  half non-empty: 547


,participant_id,target_category,utterance_masked
0,AD001,crab,"It has two claws and two legs, and it is usual..."
1,AD001,whale,Um it looks like a big creature and it has a l...
2,AD001,turtle,It has a hard shell and four legs which helps ...


Here, as a first pass, we do not remove filler words. `remove_fillers` in the CLIP branch existed because CLIP only had a token budget of 77 tokens, so filler words like "um" or "uh" pushed CLIP towards the token limit. But I think the filler words are developmentally meaningful and the Gemini embeddings here do not discard them (although, still masked). Uncomment below to remove fillers

In [3]:
# import re
# def remove_fillers(t):
#     if pd.isna(t): return t
#     t = re.sub(r'\b(um+|umm+|uh+|uhh+|okay)\b[\.,;:!?"]*', '', str(t), flags=re.I)
#     return re.sub(r'\s+', ' ', t).strip()
# utt[FULL_COL] = utt[FULL_COL].apply(remove_fillers)
# utt[HALF_COL] = utt[HALF_COL].apply(remove_fillers)
pass

In [4]:
%%time
utt["utterance_full_embedding"] = ge.embed_texts(utt[FULL_COL].tolist(), dim=DIM)
utt["utterance_half_embedding"] = ge.embed_texts(utt[HALF_COL].tolist(), dim=DIM)

print("full embedded:", utt["utterance_full_embedding"].notna().sum())
print("half embedded:", utt["utterance_half_embedding"].notna().sum())

  texts: 647 total, 0 to fetch, 547 cached
  texts: 647 total, 0 to fetch, 547 cached
full embedded: 547
half embedded: 547
CPU times: user 340 ms, sys: 112 ms, total: 452 ms
Wall time: 594 ms


In [5]:
#Add age and adult/child flag
ages = pd.read_csv(AGES_CSV).rename(columns={"participant_id_clean": "participant_id"})
utt = utt.merge(ages[["participant_id", "age_months"]], on="participant_id", how="left")
utt["is_adult"] = utt["participant_id"].str.upper().str.startswith("AD")

save = utt[["participant_id", "age_months", "target_category", "is_adult",
            FULL_COL, HALF_COL,
            "utterance_full_embedding", "utterance_half_embedding"]].copy()
for c in ["utterance_full_embedding", "utterance_half_embedding"]:
    save[c] = save[c].apply(lambda v: None if v is None else np.asarray(v).tolist())

save.to_csv(OUT_CSV, index=False)
print(f"saved {len(save)} rows -> {OUT_CSV}")
print(save["is_adult"].value_counts().rename({False: "child", True: "adult"}))

saved 647 rows -> ../data/embeddings/gemini/utterance_gemini_embeddings.csv
is_adult
child    527
adult    120
Name: count, dtype: int64


## How much did the chunking actually matter?

Counts utterances that exceeded CLIP's 77-token window and would therefore have been
chunk-averaged, and checks whether that correlates with age. If it does, the CLIP text
results carried a length artifact and the Gemini numbers are not just a re-estimate of
the same quantity.

In [6]:
def approx_clip_tokens(t):
    return np.nan if pd.isna(t) else len(str(t).split()) * 1.3  # rough BPE inflation


utt["approx_tokens"] = utt[FULL_COL].apply(approx_clip_tokens)
utt["was_chunked"] = utt["approx_tokens"] > 75

kids = utt[~utt["is_adult"] & utt["approx_tokens"].notna()]
print(f"utterances over CLIP's 77-token window: "
      f"{int(utt['was_chunked'].sum())}/{int(utt['approx_tokens'].notna().sum())} "
      f"({100 * utt['was_chunked'].mean():.1f}%)")

print("\nchunking rate by child age tercile:")
print(kids.assign(bin=pd.qcut(kids["age_months"], 3))
          .groupby("bin", observed=True)["was_chunked"].mean().round(3))

valid = kids.dropna(subset=["age_months", "approx_tokens"])
if len(valid) > 2:
    r = np.corrcoef(valid["age_months"], valid["approx_tokens"])[0, 1]
    print(f"\ncorr(age_months, utterance length) = {r:.3f}")
    if abs(r) > 0.2 and utt["was_chunked"].mean() > 0.05:
        print("  -> length is age-linked AND chunking was common: the CLIP text")
        print("     estimates were length-biased. Treat Gemini as a correction,")
        print("     not merely a different model.")
print("\ncache:", ge.cache_stats())

utterances over CLIP's 77-token window: 42/547 (6.5%)

chunking rate by child age tercile:
bin
(30.999, 71.0]    0.051
(71.0, 98.0]      0.000
(98.0, 153.0]     0.121
Name: was_chunked, dtype: float64

corr(age_months, utterance length) = 0.282
  -> length is age-linked AND chunking was common: the CLIP text
     estimates were length-biased. Treat Gemini as a correction,
     not merely a different model.

cache: {'vectors': 2271, 'json': 989, 'mb': 7.5}


## Cross-modal recognizability (utterances)

The utterance parallel of the `gemini_xmodal_cossim_*` block in
`03g_draw_generate_gemini_embedding.ipynb`: how similar are children's utterances to gemini's label of that target category?

Two differences from the drawing version, both worth keeping in mind:

- The template is `"a description of a {}"` rather than `"a drawing of a {}"`, so
  the label vectors are **not** the same six used for drawings.
- For drawings this comparison is genuinely cross-modal (image vs text). Here both
  sides are text, so the cosines sit much higher — roughly 0.46-0.81 against the
  drawings' 0.23-0.50. The two matrices are therefore **not** on a comparable
  scale; only the diagonal-vs-off-diagonal pattern within each is comparable.

In [7]:
XMODAL_OUT = OUT_DIR / "talk_gemini_xmodal.csv"

labels = ge.label_embeddings(SEA_ANIMALS, template="a description of a {}", dim=DIM)
print("label vectors:", {k: len(v) for k, v in labels.items()})

# utt still holds real arrays here: cell above converted to lists only on the
# `save` copy, not on utt itself
xm = utt.copy()
xm["target_category"] = xm["target_category"].str.lower().str.strip()
xm = xm[xm["target_category"].isin(SEA_ANIMALS)]
xm = xm[xm["utterance_full_embedding"].notna()]
print(f"{len(xm)} utterances across the six sea animals with a full embedding")

rows = []
for _, r in xm.iterrows():
    sims = {f"gemini_xmodal_cossim_{c}": ge.cosine(r["utterance_full_embedding"], labels[c])
            for c in SEA_ANIMALS}
    best = max(SEA_ANIMALS, key=lambda c: sims[f"gemini_xmodal_cossim_{c}"])
    rows.append({
        "participant_id": r["participant_id"],
        "target_category": r["target_category"],
        "age_months": r["age_months"],
        "is_adult": r["is_adult"],
        "gemini_xmodal_best": best,
        "gemini_xmodal_correct": int(best == r["target_category"]),
        **sims,
    })

talk_xmodal = pd.DataFrame(rows)
talk_xmodal.to_csv(XMODAL_OUT, index=False)
print(f"saved {len(talk_xmodal)} rows -> {XMODAL_OUT}")

kids = talk_xmodal[~talk_xmodal["is_adult"]]
print("\nchild top-1 accuracy:", round(kids["gemini_xmodal_correct"].mean(), 3))
print("adult top-1 accuracy:",
      round(talk_xmodal[talk_xmodal["is_adult"]]["gemini_xmodal_correct"].mean(), 3))
print("\naccuracy by age tercile:")
print(kids.assign(bin=pd.qcut(kids["age_months"], 3))
          .groupby("bin", observed=True)["gemini_xmodal_correct"].mean().round(3))

label vectors: {'crab': 768, 'octopus': 768, 'seahorse': 768, 'shark': 768, 'turtle': 768, 'whale': 768}
547 utterances across the six sea animals with a full embedding
saved 547 rows -> ../data/embeddings/gemini/talk_gemini_xmodal.csv

child top-1 accuracy: 0.784
adult top-1 accuracy: 0.888

accuracy by age tercile:
bin
(30.999, 71.0]    0.701
(71.0, 98.0]      0.791
(98.0, 153.0]     0.871
Name: gemini_xmodal_correct, dtype: float64
